# DecisionTune quickstart

DecisionTune 1.0 is a 395M decision model. You give it a state, a question and options. It selects one option and gives a probability for each option. You can also ask a yes/no question and get P(yes). It does one encoder pass and does not generate text, so it runs on a plain CPU. This notebook installs it, makes two single decisions, and then runs a recipe over a small CSV of support tickets.

The first run downloads the model (1.58 GB, Apache-2.0) from Hugging Face. Allow a few minutes.

In [ ]:
!pip install -q decision-tune

In [ ]:
import os
# Colab has no terminal to answer the download prompt, so consent up front.
os.environ["DECISION_TUNE_YES"] = "1"

from decision_tune import DecisionModel
m = DecisionModel.from_pretrained("decision-tune/decisiontune-1.0")

## 1. Choose one option

Route a support message to a team. Each option has a short description.

In [ ]:
m.choose("The order arrived broken.", "Which team should handle this customer message?",
         {"billing": "Billing: charges, refunds, invoices",
          "shipping": "Shipping: delivery, lost or damaged packages",
          "tech": "Tech support: bugs, crashes, login problems"})

## 2. Yes or no

You get P(yes), a float between 0 and 1.

In [ ]:
m.yes_no("The order arrived broken. I want my money back.", "Is the customer asking for a refund?")

## 3. Run a recipe over a CSV

A recipe lists the columns to read, the questions to ask, and when to flag a row for a person. `support-triage` is built in. It asks three questions (team, refund, tone).

In [ ]:
import csv
from decision_tune import Recipe

with open("tickets.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["subject", "message"])
    w.writeheader()
    w.writerows([
        {"subject": "Broken mug", "message": "The order arrived broken. I want my money back."},
        {"subject": "Charged twice", "message": "I was charged twice this month."},
        {"subject": "Cannot log in", "message": "The app crashes every time I try to log in."},
    ])

with open("tickets.csv", newline="") as f:
    rows = list(csv.DictReader(f))

results = Recipe.load("support-triage").run(rows)   # one dict per row
for r in results:
    print(r["subject"], "->", r["team"], r["refund"], r["tone"], "| needs_review:", r["needs_review"])

> **Tip: describe your options.** Short descriptions ("Shipping: delivery, lost or damaged packages") give much better results than labels only ("shipping"). Vague questions with no criteria, such as "Is this urgent?", give weak results. Write down the criteria.

Rows with `needs_review` set have a confidence below the recipe's `review_below`, or failed. A person should check those.

## Links

- [Website](https://decisiontune.com)
- [Model card](https://huggingface.co/decision-tune/decisiontune-1.0)
- [GitHub](https://github.com/decision-tune/decision-tune)